# Analysis Template: Michaelis–Menten
Testing for units support, 12/15/25

Updated 9/17/25, Nicholas Freitas

Data structures are in htbam_db_api. 
We need to make sure that transforms, fitting, masking work with units

Steps:
- Add units array to data structure (x)
- Add units to transforms (x)
- Add units to fitting  (x)
- Add units to masking (should work out of the box?) (x?)


# Outstanding work:

Checked over with Jacob.
Data looks good (our fitting here is much better, which explains the discrepancy).
May need to add auto window-finding in the fitting function.

Current mystery: my kcat stdev's are much higher than Jacob's. Can't find a reason why right now... I'll have to dig into the code and look at his data + notebook. I'm asking for his notebook and conda env file so I can replicate his results.

Next I'll need to add the new z-score filtering, background subtraction, PDF output, and new CSV output:

CSV with average initial rates per-chamber, with stdevs and number of replicates after filtering. (list positions)


In [1]:
#enables autoreloding of modules
%load_ext autoreload
%autoreload 2

from htbam_db_api.htbam_db_api import LocalHtbamDBAPI
from htbam_analysis.analysis.experiment import HTBAMExperiment
from htbam_analysis.analysis import plot
from htbam_db_api.units import units

#enable inline plotting of matplotlib figures
%matplotlib inline

#set the figure format to SVG
%config InlineBackend.figure_format = 'svg'

In [ ]:
# Difference in initial rate finding?
# Cut off beginning
# Find best possible window for linear fit
# 

## 1. Connect DB Api

In [194]:
### PARAMETERS:
EGFP_SLOPE = 91900.03 / 9 * (units.RFU / units.nM) # Ruensern correction factor (RCF)

#EGFP_SLOPE_CONC_UNITS = 'nM' #RFU/nM

root = '../example_data/9_17_25/mm_data/'
db_conn = LocalHtbamDBAPI(
    standard_curve_data_path= root + 'd3_6_StandardSeries_Analysis.csv',
    standard_name="pbp", 
    standard_substrate="pbp", 
    standard_units=units.uM,
    kinetic_data_path= root+ 'd3_TitrationSeries_Analysis.csv',
    kinetic_name="LMET", 
    kinetic_substrate="LMET", 
    kinetic_units=units.uM,
    time_units=units.s)

htbam_experiment = HTBAMExperiment(db_conn)


Connected to database.
Experiment found with the following runs:
['pbp', 'LMET', 'button_quant']


/home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_db_api/src/htbam_db_api/htbam_db_api.py:100: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.



## 2. Enzyme Quant

In [195]:
from htbam_analysis.analysis.transform import transform_data

button_concentrations = transform_data(
    data_objs = [htbam_experiment.get_run('button_quant')],              # e.g. a Data2D or Data3D or Data4D instance
    expr=f'(a_luminance / slope)',             # e.g. "(luminance - intercept) / slope"
    expression_vars={'slope': EGFP_SLOPE},
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

htbam_experiment.set_run('enzyme_concentrations', button_concentrations)     # save the fit results

Existing run data not found. Fetching from database.
Transforming 1 objects of type [<class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 1)].
Evaluating expression: (a_luminance / slope)


In [196]:
plot.plot_enzyme_concentration_chip(htbam_experiment, analysis_name='enzyme_concentrations')

## 2. Product Standards

In [197]:
from htbam_analysis.analysis.fit import fit_luminance_vs_concentration

# For demonstration purposes, I'm doing this in 3 lines
# we can either keep it "exposed" like this (might be useful?)
# Or provide a simple wrapper fx that looks like:
# htbam_experiment.fit_standard_curve(experiment_name = 'standard_5-FAM', analysis_name='htbam_experiment')

standard_experiment_data = htbam_experiment.get_run('pbp')       # retrieve our raw data
standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
htbam_experiment.set_run('pbp_standard', standard_fits)              # save the fit results

Existing run data not found. Fetching from database.
Fit slopes for 1792 wells.
Elapsed 0.998 seconds.


In [198]:
from htbam_analysis.analysis.fit import fit_luminance_vs_concentration
from htbam_analysis.analysis.filter import make_custom_mask
import numpy as np

standard_experiment_data = htbam_experiment.get_run('pbp')       # retrieve our raw data
standard_experiment_data.dep_var.shape # (8,1, 1792, 1)

# let's mask out the final concentration
mask = np.ones(standard_experiment_data.dep_var.shape) * True
mask[-1, :,:,:] = False
mask = mask.astype(bool)
custom_mask = make_custom_mask(standard_experiment_data, mask, info='final_timepoint_mask')
htbam_experiment.set_run('final_timepoint_mask', custom_mask)

htbam_experiment.apply_mask(run_name='pbp', 
                            dep_variables = ['luminance'], 
                            save_as = 'pbp_masked',
                            mask_names = ['final_timepoint_mask'])

standard_experiment_data = htbam_experiment.get_run('pbp_masked')       # retrieve our raw data
standard_fits = fit_luminance_vs_concentration(standard_experiment_data)    # perform a fit
htbam_experiment.set_run('pbp_standard', standard_fits)              # save the fit results


Fit slopes for 1792 wells.
Elapsed 0.879 seconds.


In [199]:
plot.plot_standard_curve_chip(htbam_experiment, 'pbp_standard', 'pbp_masked')

## 3. Fit Initial Rates

In [200]:
# Calculate product concentrations from RFU data:
product_concentrations = transform_data(
    data_objs = [htbam_experiment.get_run('LMET'), htbam_experiment.get_run('pbp_standard')],
    expr=f'(a_luminance - b_intercept) / b_slope',             # e.g. "(luminance - intercept) / slope"
    output_name='concentration'       # name of the new field, e.g. "concentration"
)

htbam_experiment.set_run('kinetics_LMET_conc', product_concentrations)

Existing run data not found. Fetching from database.
Transforming 2 objects of type [<class 'htbam_db_api.data.Data4D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(10, 20, 1792, 1), (1792, 3)].
Evaluating expression: (a_luminance - b_intercept) / b_slope


In [247]:
from htbam_analysis.analysis.fit import fit_concentration_vs_time

kinetics_concentrations = htbam_experiment.get_run('kinetics_LMET_conc')       # retrieve our raw data
kinetics_fits = fit_concentration_vs_time(kinetics_concentrations, start_timepoint = 1, end_timepoint=50)      # perform a fit, skipping the first timepoint
htbam_experiment.set_run('kinetics_LMET_conc_fits', kinetics_fits)                # save the fit results

Fit slopes for 1792 wells at 10 concentrations.
Elapsed 0.083 seconds.


In [248]:
plot.plot_initial_rates_chip(htbam_experiment, analysis_name='kinetics_LMET_conc_fits', experiment_name='kinetics_LMET_conc')#, remove_0_point=True) # plot the initial rates

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2826: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/matplotlib/cbook.py:1398: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.



## 4. Filter initial rates

In [262]:
# These are the "Masks" we use to select which data we want.

from htbam_analysis.analysis.filter import filter_expression_cutoff, filter_initial_rates_positive_cutoff, filter_initial_rates_r2_cutoff, filter_standard_curve_r2_cutoff

kinetics_fits = htbam_experiment.get_run('kinetics_LMET_conc_fits')       # retrieve our raw data

# Initial Rates:
initial_rates_r2_mask =         filter_initial_rates_r2_cutoff(kinetics_fits, r2_cutoff=0.9) # 0.9 R2
initial_rates_positive_mask =   filter_initial_rates_positive_cutoff(kinetics_fits)          # positive slope

# Standard Curve:
standard_curve_r2_mask =        filter_standard_curve_r2_cutoff(standard_fits, kinetics_fits, r2_cutoff=0.8) # 0.9 R2

# Expression:
expression_mask =               filter_expression_cutoff(button_concentrations, kinetics_fits, expression_cutoff=2.5) # 1 nM

# Save the masks to the experiment
htbam_experiment.set_run('initial_rates_r2_mask', initial_rates_r2_mask)
htbam_experiment.set_run('initial_rates_positive_mask', initial_rates_positive_mask)
htbam_experiment.set_run('standard_curve_r2_mask', standard_curve_r2_mask)
htbam_experiment.set_run('expression_mask', expression_mask)



In [263]:
# Plot the masks (Do one at a time)
plot.plot_mask_chip(htbam_experiment, mask_name='initial_rates_r2_mask')
#plot.plot_mask_chip(htbam_experiment, mask_name='initial_rates_positive_mask')
#plot.plot_mask_chip(htbam_experiment, mask_name='standard_curve_r2_mask')
#plot.plot_mask_chip(htbam_experiment, mask_name='expression_mask')

In [264]:
# Apply the masks to the fits and save the results
htbam_experiment.apply_mask(run_name='kinetics_LMET_conc_fits', 
                            dep_variables = ['slope', 'intercept'], 
                            save_as = 'kinetics_LMET_conc_fits_masked',
                            mask_names = ['initial_rates_r2_mask', 'initial_rates_positive_mask', 'expression_mask'])


In [265]:
htbam_experiment.get_run("kinetics_LMET_conc_fits_masked").dep_var_type

['slope', 'intercept', 'r_squared']

In [266]:
plot.plot_initial_rates_chip(experiment=htbam_experiment, analysis_name='kinetics_LMET_conc_fits_masked', experiment_name='kinetics_LMET_conc')#, remove_0_point=True) # plot the initial rates

In [267]:
plot.plot_initial_rates_vs_concentration_chip(experiment=htbam_experiment, analysis_name='kinetics_LMET_conc_fits_masked') # plot the initial rates

## 5. Fit MM Constant:

In [268]:
# I've written MM here, but it will be IC50
from htbam_analysis.analysis.fit import fit_initial_rates_vs_concentration_with_function, mm_model, inhibition_model

kinetics_fits_masked = htbam_experiment.get_run('kinetics_LMET_conc_fits_masked')       # retrieve our raw data
MM_fits, MM_pred_data = fit_initial_rates_vs_concentration_with_function(data = kinetics_fits_masked,
                                model_func = mm_model)
htbam_experiment.set_run('kinetics_LMET_MM_fits', MM_fits)                # save the fit results
htbam_experiment.set_run('kinetics_LMET_MM_pred_data', MM_pred_data)    # save the predicted data

Successfully fit nonlinear model for 1174 wells.
Elapsed 0.233 seconds.


In [269]:
from htbam_analysis.analysis.filter import filter_r2_cutoff
# Filter out crappy R2 values: 
MM_fits = htbam_experiment.get_run('kinetics_LMET_MM_fits')
MM_fits_mask = filter_r2_cutoff(MM_fits, r2_cutoff=0.8)  # 0.8 R2

htbam_experiment.set_run('MM_R2_mask', MM_fits_mask)  # save the fit results

plot.plot_mask_chip(experiment=htbam_experiment, mask_name='MM_R2_mask')

In [270]:
# Apply the mask to the fits and save the results:
htbam_experiment.apply_mask(run_name='kinetics_LMET_MM_fits', 
                            dep_variables = ['v_max', 'K_m', 'r_squared'], 
                            save_as = 'kinetics_LMET_MM_fits_masked',
                            mask_names = ['MM_R2_mask'])

In [271]:
plot.plot_MM_chip(experiment=htbam_experiment, analysis_name='kinetics_LMET_conc_fits_masked', 
                                model_fit_name='kinetics_LMET_MM_fits_masked',
                                model_pred_data_name='kinetics_LMET_MM_pred_data',
                                )#x_log=True) # plot the initial rates

# 6. Divide by [E] to get kcat, V_0/([E])
WIP

In [272]:
htbam_experiment.get_run('kinetics_LMET_conc_fits_masked').dep_var_type

['slope', 'intercept', 'r_squared']

In [273]:
# This is the same data we made above, we're just going to 
# divide everything by enzyme concentration basically

# enzyme concentration:
enzyme_concentrations = htbam_experiment.get_run("enzyme_concentrations")

# masked slopes vs conc:
masked_slopes_vs_conc = htbam_experiment.get_run("kinetics_LMET_conc_fits_masked")
# MM masked fits:
masked_fits = htbam_experiment.get_run("kinetics_LMET_MM_fits_masked")
# pred data from MM fits:
pred_data = htbam_experiment.get_run("kinetics_LMET_MM_pred_data")

# Get V_0/[E]:
masked_V0_div_E_vs_conc = transform_data(
    data_objs = [masked_slopes_vs_conc, enzyme_concentrations],
    expr=f'a_slope / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='V0_div_E'       # name of the new field, e.g. "concentration"
)

# Get V_max/[E] (kcat)
masked_fits_with_kcat = transform_data(
    data_objs = [masked_fits, enzyme_concentrations],
    expr=f'a_v_max / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='kcat',       # name of the new field, e.g. "concentration",
    keep_existing = True
)

# # Do the same for pred data:
pred_data_div_E = transform_data(
    data_objs = [pred_data, enzyme_concentrations],
    expr=f'a_y_pred / b_concentration',             # e.g. "(luminance - intercept) / slope"
    output_name='pred_V0_div_E',       # name of the new field, e.g. "concentration",
)

# Add to experiment
htbam_experiment.set_run('masked_V0_div_E_vs_conc', masked_V0_div_E_vs_conc)
htbam_experiment.set_run('masked_fits_with_kcat', masked_fits_with_kcat)
htbam_experiment.set_run('pred_data_div_E', pred_data_div_E)

Transforming 2 objects of type [<class 'htbam_db_api.data.Data3D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(10, 1792, 3), (1792, 1)].
Evaluating expression: a_slope / b_concentration
Transforming 2 objects of type [<class 'htbam_db_api.data.Data2D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(1792, 3), (1792, 1)].
Evaluating expression: a_v_max / b_concentration
Transforming 2 objects of type [<class 'htbam_db_api.data.Data3D'>, <class 'htbam_db_api.data.Data2D'>] with shapes [(10, 1792, 1), (1792, 1)].
Evaluating expression: a_y_pred / b_concentration


In [274]:
# Expects 'V0_div_E'
plot.plot_MM_div_E_chip(experiment=htbam_experiment, analysis_name='masked_V0_div_E_vs_conc', 
                                model_fit_name='masked_fits_with_kcat',
                                dep_var_name='V0_div_E',
                                )

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1878: RuntimeWarning:

Degrees of freedom <= 0 for slice.



## 7. Export to CSV

In [20]:
htbam_experiment.export_run_data_raw(run_name='kinetics_LMET_MM_fits_masked')
htbam_experiment.export_run_data_processed(run_name='kinetics_LMET_MM_fits_masked')

Run data exported to /home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_notebooks/htbam_notebooks/kinetics_LMET_MM_fits_masked_data.csv
Processed run data exported to /home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_notebooks/htbam_notebooks/kinetics_LMET_MM_fits_masked_processed_data.csv


/home/nfreitas/workspace/htbam_refactor_12_10_25/htbam_analysis/src/htbam_analysis/analysis/experiment.py:187: RuntimeWarning:

Mean of empty slice

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1878: RuntimeWarning:

Degrees of freedom <= 0 for slice.



In [21]:
# This takes long!
# Saving the figure takes around 2-5 minutes...
htbam_experiment.export_mm_subplots(
                           analysis_name='kinetics_LMET_conc_fits_masked',
                           model_fit_name='kinetics_LMET_MM_fits_masked',
                           model_pred_data_name = 'kinetics_LMET_MM_pred_data',
                           export_path = 'mm_subplots.pdf')

  0%|          | 0/56 [00:00<?, ?it/s]/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2826: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/matplotlib/cbook.py:1398: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/ma/core.py:2358: UnitStrippedWarning:

The unit of the quantity is stripped when downcasting to ndarray.

/home/nfreitas/miniconda3/envs/htbam_refactor_12_10_25/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1583: RuntimeWarning:

All-NaN slice encountered

100%|██████████| 56/56 [00:12<00:00,  4.33it/s]


Exported MM subplots to mm_subplots.pdf
